In [ ]:
!pip install opencv-python-headless

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import numpy as np
from torchvision import transforms
from PIL import Image
from IPython.display import display, Javascript
from google.colab.output import eval_js
import base64
import io

In [ ]:
# Install required packages
!pip install timm opencv-python ipywidgets
import torch
import timm
import torchvision.transforms as transforms
import cv2
from PIL import Image
from IPython.display import display
import ipywidgets as widgets
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.0 MB/s eta 0:00:00


In [ ]:
######inference######
import torch
import timm
modelm = timm.create_model(
    "efficientnet_b0",
    pretrained=False,
    num_classes=7
)
state_dict = torch.load(
    "/content/drive/MyDrive/fer_best_model_pukka.pth",
    map_location="cpu"
)
modelm.load_state_dict(state_dict)
modelm.eval()
print("✅ MODEL LOADED SUCCESSFULLY")

✅ MODEL LOADED SUCCESSFULLY


In [ ]:
# ============================================================
# FINAL PIPELINE: Webcam → modelM → VA → DB → Play + NEXT
# ============================================================
import sqlite3
import numpy as np
import cv2
import matplotlib.pyplot as plt
from IPython.display import Audio, display, Javascript
from base64 import b64decode
import torch
from torchvision import transforms
from google.colab.output import eval_js, register_callback
DB_PATH = "/content/drive/MyDrive/playlistED-DPT"
# -------------------------------
# Image preprocessing
# -------------------------------
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelm.to(device)
modelm.eval()
emotion_labels = {
    0:'angry', 1:'disgust', 2:'fear',
    3:'happy', 4:'sad', 5:'surprise', 6:'neutral'
}
# -------------------------------
# Webcam capture
# -------------------------------
def capture_image():
    js = Javascript("""
    async function takePhoto() {
        const div = document.createElement('div');
        const video = document.createElement('video');
        const btn = document.createElement('button');
        btn.innerText = "📸 Capture";
        btn.style.padding = "10px 20px";
        btn.style.fontSize = "18px";
        div.appendChild(video);
        div.appendChild(btn);
        document.body.appendChild(div);
        const stream = await navigator.mediaDevices.getUserMedia({video:true});
        video.srcObject = stream;
        await video.play();
        await new Promise(resolve => btn.onclick = resolve);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video,0,0);
        const data = canvas.toDataURL('image/jpeg');
        stream.getTracks().forEach(t => t.stop());
        div.remove();
        return data;
    }
    """)
    display(js)
    data = eval_js("takePhoto()")

    img_bytes = b64decode(data.split(',')[1])
    arr = np.frombuffer(img_bytes, np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)
# -------------------------------
# Mood detection
# -------------------------------
def detect_mood(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    x = transform(rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = modelm(x).argmax(1).item()
    return emotion_labels[pred]
# -------------------------------
# Mood → VA (0–1)
# -------------------------------
MOOD_TO_VA = {
    "happy": (0.85,0.75),
    "sad": (0.25,0.30),
    "angry": (0.20,0.85),
    "neutral": (0.50,0.50),
    "fear": (0.15,0.80),
    "surprise": (0.70,0.90),
    "disgust": (0.10,0.20)
}
def scale_to_db(x):
    return 2*x - 1
# -------------------------------
# Fetch sorted songs
# -------------------------------
def get_sorted_songs(val, aro):
    conn = sqlite3.connect("/content/drive/MyDrive/music_valence_arousal.db")
    cur = conn.cursor()
    cur.execute("""
        SELECT mp3_path, valence, arousal,
        ((valence - ?) * (valence - ?) +
         (arousal - ?) * (arousal - ?)) AS dist
        FROM predictions
        ORDER BY dist ASC
    """, (val,val,aro,aro))
    rows = cur.fetchall()
    conn.close()
    return rows
# -------------------------------
# Globals
# -------------------------------
sorted_song_list = []
song_index = 0
# -------------------------------
# NEXT SONG
# -------------------------------
def play_next_song_callback(data=None):
    global song_index
    song_index += 1
    if song_index >= len(sorted_song_list):
        print("🚫 No more songs.")
        return
    path, v, a, _ = sorted_song_list[song_index]
    song_name = path.split("/")[-1]
    print(f"\n▶ NEXT SONG ({song_index+1})")
    print("Song:", song_name)
    print("Valence:", round(v,3), "Arousal:", round(a,3))
    display(Audio(path, autoplay=True))
register_callback("play_next_song", play_next_song_callback)
# -------------------------------
# MAIN PIPELINE
# -------------------------------
def run_full_music_flow():
    global sorted_song_list, song_index
    song_index = 0
    # STEP 1: Capture
    frame = capture_image()
    # SHOW FRAME
    plt.figure(figsize=(4,4))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title("📸 Captured Frame")
    plt.axis("off")
    plt.show()
    # STEP 2: Mood
    mood = detect_mood(frame)
    print("Detected Mood:", mood)
    # STEP 3: VA
    v, a = MOOD_TO_VA.get(mood,(0.5,0.5))
    v, a = scale_to_db(v), scale_to_db(a)
    print("Mapped VA:", round(v,3), round(a,3))
    # STEP 4: Songs
    sorted_song_list = get_sorted_songs(v, a)
    TOP_K = 5
    choice = np.random.randint(0, min(TOP_K, len(sorted_song_list)))
    path, vdb, adb, _ = sorted_song_list[choice]
    song_name = path.split("/")[-1]
    print("\n🎵 NOW PLAYING")
    print("Song:", song_name)
    print("Valence:", round(vdb,3), "Arousal:", round(adb,3))
    display(Audio(path, autoplay=True))
    display(Javascript("""
        const btn = document.createElement('button');
        btn.innerText = "⏭ Next Song";
        btn.style.padding = "10px 20px";
        btn.style.fontSize = "18px";
        btn.style.marginTop = "10px";
        btn.onclick = () =>
            google.colab.kernel.invokeFunction('play_next_song', [], {});
        document.body.appendChild(btn);
    """))

In [ ]:
 run_full_music_flow()

NameError: name 'run_full_music_flow' is not defined